In [ ]:
import sys
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from typing import Annotated
from operator import add
from dotenv import load_dotenv

load_dotenv()


# ── State ─────────────────────────────────────────────────────────

class AgentState(BaseModel):
    user_input: str = ""
    category: str = ""
    thinking: Annotated[list[str], add] = []
    response: str = ""


# ── Context ───────────────────────────────────────────────────────

class UserContext(BaseModel):
    user_name: str = "Carlos"
    tone: str = "casual"
    language: str = "pt-br"


context = UserContext(user_name="Carlos", tone="casual", language="pt-br")


# ── LLM ───────────────────────────────────────────────────────────

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


# ── Nós ───────────────────────────────────────────────────────────

def classify_node(state: AgentState) -> dict:
    print("\n" + "─" * 55)
    print("🏷️  CLASSIFICANDO pergunta...")
    print("─" * 55, flush=True)

    result = llm.invoke([
        SystemMessage(content="Classifique a pergunta em UMA palavra: tecnologia, saude, financas ou geral."),
        HumanMessage(content=state.user_input),
    ])
    category = result.content.strip().lower()
    print(f"   → Categoria: '{category}'", flush=True)

    return {
        "category": category,
        "thinking": [f"🏷️ Classificação: '{category}'"],
    }


def reason_node(state: AgentState) -> dict:
    print("\n" + "─" * 55)
    print("🧠  RACIOCINANDO... (streaming token a token)")
    print("─" * 55, flush=True)

    full_content = ""
    for chunk in llm.stream([
        SystemMessage(content=(
            f"Você é um assistente especializado em {state.category}.\n"
            f"Contexto do usuário: {context.model_dump()}\n\n"
            "Antes de responder, pense passo a passo sobre a pergunta. "
            "Escreva APENAS seu raciocínio interno, não a resposta final."
        )),
        HumanMessage(content=state.user_input),
    ]):
        token = chunk.content
        print(token, end="", flush=True)
        full_content += token

    print()  # quebra de linha ao fim do stream
    return {
        "thinking": [f"🧠 Raciocínio:\n{full_content}"],
    }


def respond_node(state: AgentState) -> dict:
    print("\n" + "─" * 55)
    print("✍️   GERANDO RESPOSTA... (streaming token a token)")
    print("─" * 55, flush=True)

    reasoning = "\n".join(state.thinking)
    full_response = ""
    for chunk in llm.stream([
        SystemMessage(content=(
            f"Você é um assistente com tom {context.tone} para {context.user_name}.\n"
            f"Categoria: {state.category}\n\n"
            f"Seu raciocínio prévio:\n{reasoning}\n\n"
            "Agora escreva a resposta final para o usuário, "
            "clara e bem estruturada, baseada no seu raciocínio."
        )),
        HumanMessage(content=state.user_input),
    ]):
        token = chunk.content
        print(token, end="", flush=True)
        full_response += token

    print()
    return {
        "response": full_response,
        "thinking": ["✅ Resposta gerada com sucesso"],
    }


# ── Grafo ──────────────────────────────────────────────────────────

graph = StateGraph(AgentState)

graph.add_node("classify", classify_node)
graph.add_node("reason", reason_node)
graph.add_node("respond", respond_node)

graph.add_edge(START, "classify")
graph.add_edge("classify", "reason")
graph.add_edge("reason", "respond")
graph.add_edge("respond", END)

app = graph.compile()


# ── Execução ───────────────────────────────────────────────────────

print("=" * 55)
print("🤖 AGENTE INICIADO")
print("=" * 55)

result = app.invoke({
    "user_input": "Quais são as melhores práticas para dormir melhor?"
})

print("\n" + "=" * 55)
print("✅ PIPELINE COMPLETO")
print("=" * 55)
